In [ ]:
# Open the latest syt_record_{DATE}.csv file to pandas dataframe. Date is of the format YYYYMMDD
import pandas as pd
import glob
import os
from datetime import datetime
import sys
import re

def open_latest_syt_record_csv():
    # Get the current working directory
    cwd = os.getcwd()
    
    # Find all files matching the pattern syt_record_*.csv
    file_pattern = os.path.join(cwd, 'syt_record_*.csv')
    files = glob.glob(file_pattern)
    
    if not files:
        print("No files found matching the pattern syt_record_*.csv")
        sys.exit(1)
    
    # Extract dates from filenames and find the latest one
    date_pattern = re.compile(r'syt_record_(\d{8})\.csv')
    dated_files = []
    
    for file in files:
        match = date_pattern.search(os.path.basename(file))
        if match:
            date_str = match.group(1)
            try:
                date = datetime.strptime(date_str, '%Y%m%d')
                dated_files.append((date, file))
            except ValueError:
                print(f"Invalid date format in filename: {file}")
    
    if not dated_files:
        print("No valid dated files found.")
        sys.exit(1)
    
    # Sort files by date and get the latest one
    latest_file = max(dated_files, key=lambda x: x[0])[1]
    
    # Read the latest file into a pandas dataframe
    # Drop the first row, which is a meaningless identifier for the file
    df = pd.read_csv(latest_file, skiprows=1)
    # drop columns that are all NaN
    df = df.dropna(axis=1, how='all')
    df["Name"] = df["First Name"] + " " + df["Last Name"]
    
    return df

df = open_latest_syt_record_csv()

In [ ]:
# the Training column contains Long Training Name 1 | Long Training Name 2 | ... and the Expiration Date contains corresponding Expiration Date 1 | Expiration Date 2 | ...
# Split these columns into lists
df['Training'] = df['Training'].str.split(r' \| ')
df['Expiration Date'] = df['Expiration Date'].str.split(r' \| ')

# Transform the dataframe so that each training and expiration date is its own row
df = df.explode(['Training', 'Expiration Date'])
df.head()

In [ ]:
# find only rows containing "Y01 Safeguarding Youth Training Certification"
df_syt = df[df['Training'].str.contains('Y01 Safeguarding Youth Training Certification', na=False)]
df_syt

In [ ]:
# list any unique Names which do NOT have "Y01 Safeguarding Youth Training Certification"
names_with_syt = df_syt['Name'].unique()
all_names = df['Name'].unique()
names_without_syt = [name for name in all_names if name not in names_with_syt]
print("Names without Y01 Safeguarding Youth Training Certification:")
for name in names_without_syt:
    print(name)